# Superstore Sales Analysis
**Subqueries and CTEs **

**Business Insights being addressed**
1. Which customers spend more than the average customer?
2. How do we classify every customer into High, Medium, and Low value tiers?
3. Who are the top 3 most profitable customers in each region?

**Concepts being used:**
- Subqueries — a query nested inside another query
- `HAVING` — filtering after aggregation
- CTEs — named temporary queries that make complex SQL readable
- Chaining multiple CTEs together
- Combining CTEs with Window Functions

**Dataset:** Sample Superstore (Kaggle)  
**Tools:** Python · pandas · pandasql · JupyterLab

---
## Setup — Load Data

Reloading  Superstore dataset and importing the required libraries.


In [2]:
import pandas as pd
from pandasql import sqldf

df = pd.read_csv('../Sample - Superstore.csv', encoding='latin-1')

print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

Dataset shape: 9994 rows, 21 columns


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


---
### Importance of Subqueries and CTEs

Sometimes one query is not enough. Consider this business question:


You cannot solve this in one simple query because you need two steps:
1. First calculate the average spend across all customers
2. Then compare each individual customer against that average

**Subqueries** solve this by nesting one query inside another — the inner query runs first and passes its result to the outer query. Think of it exactly like solving brackets in maths — inside out.

**CTEs** solve the same problem but write each step as a named block, making the logic far easier to read and maintain. In real analytics teams, CTEs are strongly preferred over subqueries for this reason.


---
## Query 1 — Subquery: Above Average Customers

**Business Insight:** Which customers spend more than the average customer?

**Importance:** Identifying above-average spenders lets the business prioritise
retention efforts and loyalty rewards on customers who already drive disproportionate revenue.
Losing one high-value customer costs far more than losing several low-value ones.

**SQL concepts used:**
- `HAVING` — filters results after `GROUP BY` aggregation (like `WHERE` but for grouped data)
- Nested subquery inside `HAVING` — calculates the average first, then the outer query compares each customer against it
- Two levels of nesting — the innermost query calculates per-customer sales, the middle query averages them



In [8]:
q_subquery = """
SELECT 
    "Customer Name",
    "Segment",
    "Region",
    ROUND(SUM(Sales), 2) AS customer_total_sales
FROM df
GROUP BY "Customer ID"
HAVING SUM(Sales) > (
    SELECT AVG(customer_sales)
    FROM (
        SELECT 
            "Customer ID",
            SUM(Sales) AS customer_sales
        FROM df
        GROUP BY "Customer ID"
    )
)
ORDER BY customer_total_sales DESC;
"""

result_subquery = sqldf(q_subquery, locals())
print(f'Above average customers: {len(result_subquery)} out of {df["Customer ID"].nunique()} total')
result_subquery.head(10)


Above average customers: 294 out of 793 total


,Customer Name,Segment,Region,customer_total_sales
0,Sean Miller,Home Office,South,25043.05
1,Tamara Chand,Corporate,West,19052.22
2,Raymond Buch,Consumer,East,15117.34
3,Tom Ashbrook,Home Office,East,14595.62
4,Adrian Barton,Consumer,West,14473.57
5,Ken Lonsdale,Consumer,Central,14175.23
6,Sanjit Chand,Consumer,West,14142.33
7,Hunter Lopez,Consumer,Central,12873.30
8,Sanjit Engle,Consumer,East,12209.44
9,Christopher Conant,Consumer,South,12129.07


### Insight from Query 1: Above Average Customers

- How many customers are above average — what percentage of the total customer base is this?
There are 294 out of 793 who are above average spender making around 37% of  total customers customers . Sean Miller ranked as number 1 above average spender.
  
- Which segment dominates the above-average list — Consumer, Corporate, or Home Office?
  The **Consumer segment** dominates the list.
- Which region appears most frequently among top spenders?
  The **East Region** had the top spenders.

Out of 793 total customers, 294 (37%) spend above the average of $2,896.85.
The Consumer segment dominates the above-average list, appearing 7 out of 10 
times in the top spenders. This is suggesting Consumer accounts drive the majority 
of high-value revenue despite Home Office having the best profit margin per dollar.

It is worth noticing that Sean Miller (Home Office, South) leads all customers with 25,043 in sales, 
nearly double the second highest spender Tamara Chand at 19,052. 
This level of concentration in one customer is a retention risk worth flagging.

---
## Query 2 — CTE: Customer Value Segmentation

**Business Insight:** How do we classify every customer into value tiers based on total spending?

**Importance:** Treating all customers the same is a costly mistake.
High-value customers deserve premium service and retention efforts.
Low-value customers may need different marketing strategies to increase their spend.
This classification — often called customer tiering — is used in CRM systems across every industry.

**SQL concepts used:**
- `WITH customer_spending AS (...)` — first CTE, builds a clean per-customer summary
- `WITH spending_tiers AS (...)` — second CTE, uses the first CTE as its input
- Chaining CTEs — each block builds on the previous one, like steps in a recipe
- `CASE WHEN` inside a CTE — assigns tier labels based on spending thresholds

In [9]:
q_cte = """
WITH customer_spending AS (
    SELECT 
        "Customer ID",
        "Customer Name",
        "Segment",
        "Region",
        ROUND(SUM(Sales), 2)           AS total_sales,
        ROUND(SUM(Profit), 2)          AS total_profit,
        COUNT(DISTINCT "Order ID")     AS total_orders
    FROM df
    GROUP BY "Customer ID"
),
spending_tiers AS (
    SELECT *,
        CASE
            WHEN total_sales >= 5000 THEN 'High Value'
            WHEN total_sales >= 2000 THEN 'Medium Value'
            ELSE                          'Low Value'
        END AS customer_tier
    FROM customer_spending
)
SELECT 
    customer_tier,
    COUNT(*)                        AS customer_count,
    ROUND(AVG(total_sales), 2)      AS avg_sales,
    ROUND(SUM(total_profit), 2)     AS tier_total_profit
FROM spending_tiers
GROUP BY customer_tier
ORDER BY avg_sales DESC;
"""

result_cte = sqldf(q_cte, locals())
result_cte

,customer_tier,customer_count,avg_sales,tier_total_profit
0,High Value,117,7794.52,140313.10
1,Medium Value,323,3164.31,109233.72
2,Low Value,353,1028.81,36850.32


### Insights from Query 2: Customer Value Tiers


- How many customers fall into each tier — High, Medium, Low?
  There are 117 high valuec customers who spends an average of $ 7794 with a total profit of 140313. Among all customers there are   323 medium value customers where as we have the highest low svalue spenders.
  
- Which tier generates the most total profit for the business?
  Top tier customers generate the most revenue amongst all kinds of spenders.

  
- What percentage of customers are High Value — and what does that tell you about revenue concentration?
  Around 15% of customers are high value customers. The majority of customers fall into the **Low Value** tier, yet **High Value**   customers generate a disproportionate share of total profit. This makes High Value customer retention the single most important business priority. 
  


---
## Query 3 — CTE + Window Function: Top 3 Customers Per Region

**Business Insight:** Who are the top 3 most profitable customers in each region?

**Importance:** Regional sales managers need to know who their most valuable accounts are.
These customers should receive personalised outreach, dedicated account managers,
and be the first to hear about new products. Losing a top regional customer
is a significant local revenue event.

**SQL concepts used:**
- CTE + Window Function combined — this is real production-level SQL
- `RANK() OVER (PARTITION BY Region ORDER BY total_profit DESC)` — ranks customers within each region
- `WHERE profit_rank <= 3` — filters after ranking to keep only top 3 per region
- This pattern is called **Top N per group** — one of the most common SQL interview questions


In [10]:
q_top_customers = """
WITH customer_profit AS (
    SELECT 
        "Customer Name",
        "Region",
        ROUND(SUM(Profit), 2)          AS total_profit,
        ROUND(SUM(Sales), 2)           AS total_sales,
        COUNT(DISTINCT "Order ID")     AS total_orders
    FROM df
    GROUP BY "Customer ID"
),
ranked_customers AS (
    SELECT *,
        RANK() OVER (
            PARTITION BY Region
            ORDER BY total_profit DESC
        ) AS profit_rank
    FROM customer_profit
)
SELECT *
FROM ranked_customers
WHERE profit_rank <= 3
ORDER BY Region, profit_rank;
"""

result_top = sqldf(q_top_customers, locals())
result_top

,Customer Name,Region,total_profit,total_sales,total_orders,profit_rank
0,Hunter Lopez,Central,5622.43,12873.30,6,1
1,Greg Tran,Central,2163.43,11820.12,11,2
2,Laura Armstrong,Central,2059.12,8673.22,11,3
3,Raymond Buch,East,6976.10,15117.34,6,1
4,Tom Ashbrook,East,4703.79,14595.62,4,2
5,Keith Dawkins,East,3038.63,8181.26,12,3
6,Christopher Martinez,South,3899.89,8954.02,4,1
7,Christopher Conant,South,2177.05,12129.07,5,2
8,Robert Marley,South,1902.54,5979.10,5,3
9,Tamara Chand,West,8981.32,19052.22,5,1


### Insights from  Query 3: Top Customers Per Region

- Which region has the highest-profit #1 customer?
The **West Region** has the highest-profit customers Tamara Chand despite having only 4 orders in total which means the  revenue heavily depends on this customer purchases and this should be retained and be the priority.
- Is there a big gap between rank #1 and rank #3 within any region?
There is a significant gap btween central region btween #1 and #3 rank, indicating dangerous revenue concentration in a single account.The regional manager should focus on developing new high-value relationships.
- Do any customer names appear across multiple regions — and what would that mean?
No Customr names are appearing across multiple regions. this means that buying behaviour 
is locally driven.
